In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

master = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v2.csv")

print(f"✅ Loaded: {master.shape[0]} facilities x {master.shape[1]} columns")
print(f"Districts: {master['District'].nunique()}")
print(f"Regions: {master['Region'].nunique()}")

✅ Loaded: 9978 facilities x 32 columns
Districts: 261
Regions: 16


In [2]:
# What types of facilities do we have?
print("=== FACILITY TYPES ===\n")
print(master['Facility_Type'].value_counts().to_string())
print(f"\nTotal unique types: {master['Facility_Type'].nunique()}")

=== FACILITY TYPES ===

Facility_Type
CHPS                          6732
HEALTH CENTRE                 1215
CLINIC                         929
HOSPITAL                       581
MATERNITY HOME                 248
DISTRICT HOSPITAL              145
POLYCLINIC                      93
TEACHING HOSPITAL               10
REGIONAL HOSPITAL               10
UNIVERSITY HOSPITAL/CLINIC       8
PSYCHIATRIC HOSPITAL             5
LEPROSARIUM                      2

Total unique types: 12


##### Different facility types provide different levels of care. A CHPS compound is basically a small community health post — it can handle simple things like malaria treatment, vaccinations, and basic checkups. But if you need surgery, blood transfusion, emergency care for a complicated pregnancy, or treatment for a serious condition — a CHPS compound cannot help you. You need a hospital.

##### So when we see that 6,732 out of 9,978 facilities are CHPS compounds, that tells us something important: the majority of Ghana's healthcare infrastructure can only handle basic care. The country looks like it has nearly 10,000 facilities, which sounds impressive. But when you realize 67% of them are basic CHPS compounds, the picture changes.

In [3]:
# Facilities per region + population
region_summary = (
    master.groupby('Region')
    .agg(
        Facilities=('Name', 'count'),
        Districts=('District', 'nunique'),
        Population=('District_Population', lambda x: x.drop_duplicates().sum()),
        Avg_Distance=('distance_from_centroid_km', 'mean')
    )
    .sort_values('Facilities', ascending=False)
)

region_summary['Per_10k_People'] = round(region_summary['Facilities'] / region_summary['Population'] * 10000, 2)
region_summary['Avg_Distance'] = round(region_summary['Avg_Distance'], 1)

print("=== REGIONAL OVERVIEW ===\n")
print(region_summary.to_string())

=== REGIONAL OVERVIEW ===

               Facilities  Districts  Population  Avg_Distance  Per_10k_People
Region                                                                        
ASHANTI              1645         44     5646798           9.0            2.91
GREATER ACCRA        1464         34     5455692           4.5            2.68
EASTERN              1133         39     2925653           9.7            3.87
CENTRAL               770         26     2859821          10.0            2.69
UPPER EAST            673         15     1301226          10.1            5.17
WESTERN               654         14     2060585          14.1            3.17
NORTHERN              562         16     2350343          15.4            2.39
VOLTA                 550         23     1759105          10.5            3.13
UPPER WEST            513         11      901502          15.7            5.69
BONO                  478         12     1426978          11.7            3.35
BONO EAST             404

##### After looking at each region's facilities, population, average distance, and the per 10,000 ratio. The realization was:

##### Having the most facilities doesn't mean you're best served. Ashanti has 1,645 facilities but also 5.5 million people, so the ratio is only 2.91. Upper West has only 513 facilities but a small population, so it has the best ratio at 5.69. But ratio isn't everything either, Savannah's average distance is 32km. Even if you have enough facilities, what good is it if people can't reach them?

##### So it will be wise to look at both the ratio AND the distance together to understand access.

In [4]:
# District-level: best and worst served
district_summary = (
    master.groupby(['District', 'Region'])
    .agg(
        Facilities=('Name', 'count'),
        Population=('District_Population', 'first'),
        Avg_Distance=('distance_from_centroid_km', 'mean')
    )
)

district_summary['Per_10k_People'] = round(district_summary['Facilities'] / district_summary['Population'] * 10000, 2)
district_summary['Avg_Distance'] = round(district_summary['Avg_Distance'], 1)

print("=== TOP 10 BEST SERVED DISTRICTS (most facilities per 10k people) ===\n")
print(district_summary.sort_values('Per_10k_People', ascending=False).head(10).to_string())

print("\n\n=== TOP 10 WORST SERVED DISTRICTS (fewest facilities per 10k people) ===\n")
print(district_summary.sort_values('Per_10k_People', ascending=True).head(10).to_string())

=== TOP 10 BEST SERVED DISTRICTS (most facilities per 10k people) ===

                                       Facilities  Population  Avg_Distance  Per_10k_People
District                   Region                                                          
NORTH EAST GONJA           SAVANNAH            64       39404          36.2           16.24
NANDOM                     UPPER WEST          52       51328           7.4           10.13
GOMOA EAST                 CENTRAL             84       83610          14.1           10.05
LAWRA                      UPPER WEST          57       58433           8.2            9.75
BUILSA NORTH               UPPER EAST          51       56571           9.9            9.02
DAFFIAMA-BUSSIE-ISSA       UPPER WEST          34       38754          17.0            8.77
BIRIM SOUTH                EASTERN             30       35654           8.5            8.41
SEKYERE AFRAM PLAINS NORTH ASHANTI             26       32640          30.1            7.97
BUILSA SO

In [5]:
suspects = ['AWUTU SENYA WEST', 'AWUTU SENYA EAST', 'UPPER WEST AKIM', 'WEIJA GBAWE', 'GA SOUTH']

for d in suspects:
    rows = master[master['District'] == d]
    print(f"\n{d}:")
    print(f"  Facilities: {len(rows)}")
    print(f"  Region: {rows['Region'].unique()}")
    print(f"  Population: {rows['District_Population'].unique()}")


AWUTU SENYA WEST:
  Facilities: 27
  Region: ['CENTRAL' 'GREATER ACCRA']
  Population: [161460 308697 350121 236527]

AWUTU SENYA EAST:
  Facilities: 57
  Region: ['CENTRAL' 'GREATER ACCRA']
  Population: [236527 161460 350121 308697]

UPPER WEST AKIM:
  Facilities: 29
  Region: ['EASTERN' 'GREATER ACCRA']
  Population: [ 93391 350121]

WEIJA GBAWE:
  Facilities: 98
  Region: ['GREATER ACCRA' 'CENTRAL']
  Population: [350121 213674 153490 236527 332232 159208]

GA SOUTH:
  Facilities: 13
  Region: ['GREATER ACCRA' 'CENTRAL']
  Population: [350121 161460]


In [6]:
master.head(10)

,ID,Name,Facility_Type,Ownership,Region,District,Sub-District,Community,Latitude,Longitude,has_emonc,has_midwife,has_blood_bank,District_standardized,District_Population,District_Urban_Pop,District_Rural_Pop,Percentage of Urban,Percentage of Rural,Male_Total,Female_Total,Household_Total,NonHousehold_Total,Urban_Male,Urban_Female,Rural_Male,Rural_Female,Urban_Population,Rural_Population,Centroid_Lat,Centroid_Lon,distance_from_centroid_km
0,4608,1 MEDICAL RECEPTION STATION(1MRS),POLYCLINIC,QUASI-GOVERNMENT,GREATER ACCRA,KPONE-KATAMANSO,GBETSILE,MICHEL CAMP,5.728982,-0.025255,False,False,False,KPONE KATAMANSO,417334,394882,22452,0.946201,0.053799,208040,209294,416128,1206,196707,198175,11333,11119,394882,22452,5.756751,-0.059359,4.875
1,564,2MRS MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,WESTERN,EFFIA KWESIMINTSIM,APREMDO,ALREADY BARRACKS,4.914347,-1.805559,False,False,False,EFFIA KWESIMINTSIM MUNICIPAL,173975,173975,-,1.000000,0.000000,85864,88111,170992,2983,85864,88111,-,-,173975,-,4.964357,-1.787798,5.899
2,9884,31ST DWM CHPS,CHPS,GOVERNMENT,ASHANTI,KUMASI,MANHYIA -ASH TOWN,ASH TOWN,6.705688,-1.621758,False,False,False,KUMASI METROPOLITAN,443981,443981,-,1.000000,0.000000,213662,230319,413561,30420,213662,230319,-,-,443981,-,6.688186,-1.621228,1.947
3,3955,37 MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,GREATER ACCRA,AYAWASO EAST,KANDA,<NULL>,5.588470,-0.183260,False,False,False,AYAWASO EAST MUNICIPAL,53004,53004,0,1.000000,0.000000,25438,27566,52508,496,25438,27566,NaN,NaN,53004,NaN,5.590851,-0.191775,0.979
4,4865,3E MEDICAL CENTRE,CLINIC,PRIVATE,GREATER ACCRA,WEIJA GBAWE,BORTIANOR,BORTIANOR REDTOP,5.531571,-0.351723,False,False,False,GA SOUTH,350121,266721,83400,0.761797,0.238203,172492,177629,349171,950,131068,135653,41424,41976,266721,83400,5.549603,-0.351312,2.006
5,4098,3M&C GHANA LIMITED,HEALTH CENTRE,PRIVATE,GREATER ACCRA,AYAWASO WEST,LEGON,LEGON,5.643030,-0.153326,False,False,False,AYAWASO WEST MUNICIPAL,75303,75303,0,1.000000,0.000000,38614,36689,60952,14351,38614,36689,NaN,NaN,75303,NaN,5.633869,-0.173598,2.464
6,7900,3MRS SUNYANI,HOSPITAL,QUASI-GOVERNMENT,BONO,SUNYANI,NEW DORMAA,ASUAKWAH,7.341959,-2.294543,False,False,False,SUNYANI MUNICIPAL,193595,156343,37252,0.807578,0.192422,96358,97237,185031,8564,77088,79255,19270,17982,156343,37252,7.234701,-2.364255,14.190
7,10078,3WAY FAMILY CARE CLINIC(CLOSED),CLINIC,PRIVATE,AHAFO,ASUTIFI SOUTH,ACHERENSUA,KROFROM,6.979922,-2.289259,False,False,False,ASUTIFI SOUTH,68394,33236,35158,0.485949,0.514051,34932,33462,66692,1702,16515,16721,18417,16741,33236,35158,6.837113,-2.390411,19.412
8,9130,4MRS CLINIC,CLINIC,QUASI-GOVERNMENT,ASHANTI,KUMASI,SUBIN NORTH,4BM BARRACKS,6.694661,-1.629465,False,False,False,KUMASI METROPOLITAN,443981,443981,-,1.000000,0.000000,213662,230319,413561,30420,213662,230319,-,-,443981,-,6.688186,-1.621228,1.160
9,3780,64 BENCH CHPS,CHPS,GOVERNMENT,NORTHERN,TAMALE,TAMALE CENTRAL,CHANGLI,9.395928,-0.838161,False,False,False,TAMALE METROPOLITAN,374744,374744,-,1.000000,0.000000,185051,189693,365510,9234,185051,189693,-,-,374744,-,9.373733,-0.743034,10.724


In [7]:
# How many districts differ between the two columns?
sample = master[['District', 'District_standardized']].drop_duplicates()
different = sample[sample['District'] != sample['District_standardized']]
print(f"District pairs that differ: {len(different)}")
print(f"\nFirst 15:")
print(f"{'District (GADM)':<35} {'District_standardized (old)'}")
print("="*70)
for _, row in different.head(15).iterrows():
    print(f"{row['District']:<35} {row['District_standardized']}")

District pairs that differ: 421

First 15:
District (GADM)                     District_standardized (old)
KPONE-KATAMANSO                     KPONE KATAMANSO
EFFIA KWESIMINTSIM                  EFFIA KWESIMINTSIM MUNICIPAL
KUMASI                              KUMASI METROPOLITAN
AYAWASO EAST                        AYAWASO EAST MUNICIPAL
WEIJA GBAWE                         GA SOUTH
AYAWASO WEST                        AYAWASO WEST MUNICIPAL
SUNYANI                             SUNYANI MUNICIPAL
TAMALE                              TAMALE METROPOLITAN
SAGNERIGU                           SAGNARIGU MUNICIPAL
TEMA                                TEMA METROPOLITAN
NSAWAM ADOAGYIRI                    NSAWAM ADOAGYIRI MUNICIPAL
KWADASO                             KUMASI METROPOLITAN
KWADASO                             KWADASO MUNICIPAL
TARKWA NSUAEM                       TARKWA-NSUAEM MUNICIPAL
AJUMAKO-ENYAN-ESSIAM                AJUMAKU ENYAN ESSIAM


In [8]:
# Check if we still have the population data
try:
    print(f"all_sheets available: {list(all_sheets.keys())}")
except:
    print("all_sheets is NOT in memory — we need to reload")

all_sheets is NOT in memory — we need to reload


In [9]:
# Reload the Excel workbook
all_sheets = pd.read_excel(
    r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\Version 1.4 -- Ghana Health Service Data Project - Copy.xlsx",
    sheet_name=None
)

print("Sheets loaded:")
for name, df in all_sheets.items():
    print(f"  {name}: {df.shape}")

Sheets loaded:
  raw data: (10337, 24)
  hospital-to-centroids-km: (10337, 21)
  hospitals-data: (9978, 13)
  Table2: (834, 17)
  all-districts-data: (834, 17)
  gss-population-distribution: (261, 6)
  district-centroids: (261, 4)
  not-relevant-yet: (273, 7)
  analysis: (264, 5)


In [10]:
# Step 1: Flatten demographics (same process as before, using 'District' column as join key)
all_districts = all_sheets['all-districts-data']

main_rows = all_districts[all_districts['Type'] == 'Main'].copy()
urban_rows = all_districts[all_districts['Type'] == 'Urban'].copy()
rural_rows = all_districts[all_districts['Type'] == 'Rural'].copy()

main_cols = {
    'Both Sexes [Total Population]': 'Total_Population',
    'Male [Total Population]': 'Male_Total',
    'Female [Total Population]': 'Female_Total',
    'Both Sexes [Household Population]': 'Household_Total',
    'Male [Household Population]': 'Household_Male',
    'Female [Household Population]': 'Household_Female',
    'Both Sexes [Non-household Population]': 'NonHousehold_Total',
    'Male [Non-household Population]': 'NonHousehold_Male',
    'Female [Non-household Population]': 'NonHousehold_Female',
}

urban_cols = {
    'Both Sexes [Total Population]': 'Urban_Population',
    'Male [Total Population]': 'Urban_Male',
    'Female [Total Population]': 'Urban_Female',
    'Both Sexes [Household Population]': 'Urban_Household_Total',
    'Male [Household Population]': 'Urban_Household_Male',
    'Female [Household Population]': 'Urban_Household_Female',
    'Both Sexes [Non-household Population]': 'Urban_NonHousehold_Total',
    'Male [Non-household Population]': 'Urban_NonHousehold_Male',
    'Female [Non-household Population]': 'Urban_NonHousehold_Female',
}

rural_cols = {
    'Both Sexes [Total Population]': 'Rural_Population',
    'Male [Total Population]': 'Rural_Male',
    'Female [Total Population]': 'Rural_Female',
    'Both Sexes [Household Population]': 'Rural_Household_Total',
    'Male [Household Population]': 'Rural_Household_Male',
    'Female [Household Population]': 'Rural_Household_Female',
    'Both Sexes [Non-household Population]': 'Rural_NonHousehold_Total',
    'Male [Non-household Population]': 'Rural_NonHousehold_Male',
    'Female [Non-household Population]': 'Rural_NonHousehold_Female',
}

main_renamed = main_rows.rename(columns=main_cols)[['District', 'Main District', 'Region'] + list(main_cols.values())]
urban_renamed = urban_rows.rename(columns=urban_cols)[['District'] + list(urban_cols.values())]
rural_renamed = rural_rows.rename(columns=rural_cols)[['District'] + list(rural_cols.values())]

district_demographics = main_renamed.merge(urban_renamed, on='District', how='left')
district_demographics = district_demographics.merge(rural_renamed, on='District', how='left')

# Remove sub-metro rows
sub_metros = [
    'Accra Metropolitan Ablekuma South', 'Accra Metropolitan Ashiedu Keteke',
    'Accra Metropolitan Okaikoi South', 'Kumasi Metropolitan Bantama',
    'Kumasi Metropolitan Manhyia North', 'Kumasi Metropolitan Manhyia South',
    'Kumasi Metropolitan Nhyiaeso', 'Kumasi Metropolitan Subin',
    'Cape Coast Metropolitan Cape Coast North', 'Cape Coast Metropolitan Cape Coast South',
    'Sekondi Takoradi Metropolitan Essikado Ketan', 'Sekondi Takoradi Metropolitan Sekondi',
    'Sekondi Takoradi Metropolitan Takoradi', 'Tamale Metropolitan Tamale Central',
    'Tamale Metropolitan Tamale South', 'Tema Central District', 'Tema East District',
]
district_demographics = district_demographics[~district_demographics['Main District'].isin(sub_metros)]

district_demographics['Main District'] = district_demographics['Main District'].replace({
    'Mion District District': 'Mion District',
    'Mophor (Mpohor) Mpohor (Mpohor)': 'Mophor (Mpohor) District'
})

# Step 2: Get population distribution (urban/rural percentages)
population = all_sheets['gss-population-distribution']

print(f"Demographics: {district_demographics.shape}")
print(f"Population distribution: {population.shape}")
print(f"\n✅ Source data rebuilt. Ready to re-merge.")

Demographics: (261, 30)
Population distribution: (261, 6)

✅ Source data rebuilt. Ready to re-merge.


In [11]:
# Step 2: Drop old population columns and District_standardized from master

# Keep only facility-related columns
keep_cols = ['ID', 'Name', 'Facility_Type', 'Ownership', 'Region', 'District',
             'Sub-District', 'Community', 'Latitude', 'Longitude',
             'has_emonc', 'has_midwife', 'has_blood_bank',
             'Centroid_Lat', 'Centroid_Lon', 'distance_from_centroid_km']

master_clean = master[keep_cols].copy()

print(f"Master before: {master.shape[1]} columns")
print(f"Master after dropping population cols: {master_clean.shape[1]} columns")
print(f"Facilities: {len(master_clean)}")
print(f"Unique districts: {master_clean['District'].nunique()}")

Master before: 32 columns
Master after dropping population cols: 16 columns
Facilities: 9978
Unique districts: 261


In [12]:
# What columns did we drop?
dropped = [col for col in master.columns if col not in keep_cols]

print(f"Dropped {len(dropped)} columns:\n")
for col in dropped:
    print(f"  {col}")

Dropped 16 columns:

  District_standardized
  District_Population
  District_Urban_Pop
  District_Rural_Pop
  Percentage of Urban
  Percentage of Rural
  Male_Total
  Female_Total
  Household_Total
  NonHousehold_Total
  Urban_Male
  Urban_Female
  Rural_Male
  Rural_Female
  Urban_Population
  Rural_Population


In [13]:
# The demographics data
print("=== DEMOGRAPHICS (district_demographics) ===")
print(f"Shape: {district_demographics.shape}")
print(f"\nColumns:")
for col in district_demographics.columns:
    print(f"  {col}")

print(f"\n\n=== POPULATION DISTRIBUTION (population) ===")
print(f"Shape: {population.shape}")
print(f"\nColumns:")
for col in population.columns:
    print(f"  {col}")

=== DEMOGRAPHICS (district_demographics) ===
Shape: (261, 30)

Columns:
  District
  Main District
  Region
  Total_Population
  Male_Total
  Female_Total
  Household_Total
  Household_Male
  Household_Female
  NonHousehold_Total
  NonHousehold_Male
  NonHousehold_Female
  Urban_Population
  Urban_Male
  Urban_Female
  Urban_Household_Total
  Urban_Household_Male
  Urban_Household_Female
  Urban_NonHousehold_Total
  Urban_NonHousehold_Male
  Urban_NonHousehold_Female
  Rural_Population
  Rural_Male
  Rural_Female
  Rural_Household_Total
  Rural_Household_Male
  Rural_Household_Female
  Rural_NonHousehold_Total
  Rural_NonHousehold_Male
  Rural_NonHousehold_Female


=== POPULATION DISTRIBUTION (population) ===
Shape: (261, 6)

Columns:
  District
  Main
  Urban
  Rural
  Percentage of Urban
  Percentage of Rural


In [14]:
def make_simple_key(name):
    return (
        str(name)
        .upper()
        .replace('-', ' ')
        .replace('/', ' ')
        .replace('(', '')
        .replace(')', '')
        .replace(' MUNICIPAL', '')
        .replace(' METROPOLITAN', '')
        .replace(' DISTRICT', '')
        .replace('  ', ' ')
        .strip()
    )

district_demographics['simple_key'] = district_demographics['Main District'].apply(make_simple_key)
population['simple_key'] = population['District'].apply(make_simple_key)
master_clean['simple_key'] = master_clean['District'].apply(make_simple_key)

demo_keys = set(district_demographics['simple_key'])
master_keys = set(master_clean['simple_key'])

matched = master_keys & demo_keys
unmatched = master_keys - demo_keys

print(f"Matched: {len(matched)} of {len(master_keys)}")
print(f"Unmatched: {len(unmatched)}")

if unmatched:
    print(f"\nUnmatched districts:")
    for k in sorted(unmatched):
        print(f"  {k}")

Matched: 235 of 261
Unmatched: 26

Unmatched districts:
  ADANSI AKROFUOM
  ADENTA
  AGOTIME ZIOPE
  AJUMAKO ENYAN ESSIAM
  AKWAPEM NORTH
  AKWAPEM SOUTH
  BIBIANI ANHWIASO BEKWAI
  BOLGA EAST
  BOSOMTWE
  BUNKPURUGU NAKPANDURI
  DAFFIAMA BUSSIE ISSA
  DORMAA
  EJISU
  KASENA NANKANA EAST
  KASENA NANKANA WEST
  LA DADE KOTOPON
  LA NKWANTANANG MADINA
  MFANTSEMAN
  MPOHOR
  OKAIKWEI NORTH
  SAGNERIGU
  SEKYERE AFRAM PLAINS NORTH
  TATALE SANGULI
  TWIFO ATTI MORKWA
  TWIFO HEMANG LOWER DENKYIRA
  UPPER MANYA


In [15]:
# Mapping: GADM simple_key -> Census simple_key
gadm_to_census = {
    'ADANSI AKROFUOM': 'AKROFUOM',
    'ADENTA': 'ADENTAN',
    'AGOTIME ZIOPE': 'AGORTIME ZIOPE',
    'AJUMAKO ENYAN ESSIAM': 'AJUMAKU ENYAN ESSIAM',
    'AKWAPEM NORTH': 'AKWAPIM NORTH',
    'AKWAPEM SOUTH': 'AKWAPIM SOUTH',
    'BIBIANI ANHWIASO BEKWAI': 'SEFWI BIBIANI AHWIASO BEKWAI',
    'BOLGA EAST': 'BOLGATANGA EAST',
    'BOSOMTWE': 'BOSOMTWI',
    'BUNKPURUGU NAKPANDURI': 'BUNKPURUGU NYANKPANDURI',
    'DAFFIAMA BUSSIE ISSA': 'DAFFIAMA BUSSIE',
    'DORMAA': 'DORMAA CENTRAL',
    'EJISU': 'EJISU JUABEN',
    'KASENA NANKANA EAST': 'KASSENA NANKANA',
    'KASENA NANKANA WEST': 'KASSENA NANKANA WEST',
    'LA DADE KOTOPON': 'LA DADEKOTOPON',
    'LA NKWANTANANG MADINA': 'LA NKWANTANAN MADINA',
    'MFANTSEMAN': 'MFANTSIMAN',
    'MPOHOR': 'MOPHOR MPOHOR',
    'OKAIKWEI NORTH': 'OKAI KOI NORTH',
    'SAGNERIGU': 'SAGNARIGU',
    'SEKYERE AFRAM PLAINS NORTH': 'SEKYERE AFRAM PLAINS',
    'TATALE SANGULI': 'TATALE',
    'TWIFO ATTI MORKWA': 'TWIFO ATI MORKWA',
    'TWIFO HEMANG LOWER DENKYIRA': 'TWIFO HEMAN LOWER DENKYIRA',
    'UPPER MANYA': 'UPPER MANYA KROBO',
}

# Apply mapping to master
master_clean['census_key'] = master_clean['simple_key'].map(lambda x: gadm_to_census.get(x, x))

# Also need GUAN -> map to something in census
# GUAN was carved from Hohoe, but it won't have its own census population
# Let's check
print("Is GUAN in census data?")
print('GUAN' in demo_keys)

# Check what's left unmatched
unmatched_after = set(master_clean['census_key']) - demo_keys
print(f"\nUnmatched after mapping: {len(unmatched_after)}")
if unmatched_after:
    for k in sorted(unmatched_after):
        print(f"  {k}")

Is GUAN in census data?
True

Unmatched after mapping: 1
  KASSENA NANKANA


In [16]:
# What does census have for Kassena/Kasena?
for k in sorted(demo_keys):
    if 'KASSENA' in k or 'KASENA' in k or 'NANKANA' in k:
        print(f"  Census: {k}")

  Census: KASSENA NANKANA EAST
  Census: KASSENA NANKANA WEST


In [17]:
# Fix the mapping
master_clean['census_key'] = master_clean['census_key'].replace({
    'KASSENA NANKANA': 'KASSENA NANKANA EAST'
})

# Verify
unmatched_final = set(master_clean['census_key']) - demo_keys
print(f"Unmatched: {len(unmatched_final)}")

Unmatched: 0


In [18]:
# Merge demographics
demo_cols = ['simple_key', 'Total_Population', 'Male_Total', 'Female_Total',
             'Household_Total', 'NonHousehold_Total',
             'Urban_Population', 'Urban_Male', 'Urban_Female',
             'Rural_Population', 'Rural_Male', 'Rural_Female']

master_clean = master_clean.merge(
    district_demographics[demo_cols].drop_duplicates(subset='simple_key'),
    left_on='census_key', right_on='simple_key', how='left', suffixes=('', '_drop')
)

# Merge population distribution (urban/rural percentages)
pop_cols = ['simple_key', 'Main', 'Urban', 'Rural', 'Percentage of Urban', 'Percentage of Rural']

population['simple_key_pop'] = population['District'].apply(make_simple_key)
# Apply same census mapping for population
pop_key_map = gadm_to_census.copy()
population['census_key'] = population['simple_key_pop'].map(lambda x: gadm_to_census.get(x, x))

# Actually population uses its own names, let's match via census_key on master
master_clean = master_clean.merge(
    population[['simple_key_pop', 'Percentage of Urban', 'Percentage of Rural']].rename(columns={'simple_key_pop': 'pop_key'}),
    left_on='census_key', right_on='pop_key', how='left'
)

# Clean up helper columns
drop_cols = [c for c in master_clean.columns if c in ['simple_key', 'simple_key_drop', 'census_key', 'pop_key']]
master_clean = master_clean.drop(columns=drop_cols)

# Rename Total_Population to District_Population
master_clean = master_clean.rename(columns={'Total_Population': 'District_Population'})

print(f"Master: {master_clean.shape}")
print(f"\nMissing values:")
for col in master_clean.columns:
    missing = master_clean[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing}")

print(f"\nUnique districts: {master_clean['District'].nunique()}")

Master: (9978, 29)

Missing values:
  Rural_Population: 894
  Rural_Male: 894
  Rural_Female: 894

Unique districts: 261


In [19]:
# Are the missing rural values from 100% urban districts?
missing_rural = master_clean[master_clean['Rural_Population'].isna()]['District'].unique()
print(f"Districts with missing rural data: {len(missing_rural)}\n")

for d in sorted(missing_rural):
    pct = master_clean[master_clean['District'] == d]['Percentage of Urban'].iloc[0]
    print(f"  {d}: {pct*100:.0f}% urban")

Districts with missing rural data: 18

  ABLEKUMA CENTRAL: 100% urban
  ABLEKUMA NORTH: 100% urban
  ABLEKUMA WEST: 100% urban
  ADENTA: 100% urban
  ASHAIMAN: 100% urban
  AYAWASO CENTRAL: 100% urban
  AYAWASO EAST: 100% urban
  AYAWASO NORTH: 100% urban
  AYAWASO WEST: 100% urban
  GA CENTRAL: 100% urban
  GA NORTH: 100% urban
  KORLE-KLOTTEY: 100% urban
  KROWOR: 100% urban
  LA-DADE-KOTOPON: 100% urban
  LEDZOKUKU: 100% urban
  OKAIKWEI NORTH: 100% urban
  TEMA WEST: 100% urban
  WEIJA GBAWE: 100% urban


In [20]:
# Check the districts that were problematic before
suspects = ['WEIJA GBAWE', 'GA SOUTH', 'KWADASO', 'AWUTU SENYA WEST']

for d in suspects:
    rows = master_clean[master_clean['District'] == d]
    pop = rows['District_Population'].unique()
    print(f"{d}: {len(rows)} facilities, Population: {pop}")

WEIJA GBAWE: 98 facilities, Population: [213674]
GA SOUTH: 13 facilities, Population: [350121]
KWADASO: 57 facilities, Population: [154526]
AWUTU SENYA WEST: 27 facilities, Population: [161460]


In [21]:
master = master_clean.copy()

master.to_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv", index=False)

print(f"✅ Master v3 saved: {master.shape[0]} facilities x {master.shape[1]} columns")
print(f"Districts: {master['District'].nunique()}")
print(f"Regions: {master['Region'].nunique()}")

✅ Master v3 saved: 9978 facilities x 29 columns
Districts: 261
Regions: 16


In [22]:
# Regional overview - now with correct numbers
region_summary = (
    master.groupby('Region')
    .agg(
        Facilities=('Name', 'count'),
        Districts=('District', 'nunique'),
        Population=('District_Population', lambda x: x.drop_duplicates().sum()),
        Avg_Distance=('distance_from_centroid_km', 'mean')
    )
    .sort_values('Facilities', ascending=False)
)

region_summary['Per_10k_People'] = round(region_summary['Facilities'] / region_summary['Population'] * 10000, 2)
region_summary['Avg_Distance'] = round(region_summary['Avg_Distance'], 1)

print("=== REGIONAL OVERVIEW (CORRECTED) ===\n")
print(region_summary.to_string())

=== REGIONAL OVERVIEW (CORRECTED) ===

               Facilities  Districts  Population  Avg_Distance  Per_10k_People
Region                                                                        
ASHANTI              1645         44     5531488           9.0            2.97
GREATER ACCRA        1464         34     6224145           4.5            2.35
EASTERN              1133         39     3900406           9.7            2.90
CENTRAL               770         26     3702953          10.0            2.08
UPPER EAST            673         15     1301226          10.1            5.17
WESTERN               654         14     2060585          14.1            3.17
NORTHERN              562         16     2310939          15.4            2.43
VOLTA                 550         23     2215937          10.5            2.48
UPPER WEST            513         11      901502          15.7            5.69
BONO                  478         12     1208649          11.7            3.95
BONO EAST    

In [23]:
# District-level: best and worst served
district_summary = (
    master.groupby(['District', 'Region'])
    .agg(
        Facilities=('Name', 'count'),
        Population=('District_Population', 'first'),
        Avg_Distance=('distance_from_centroid_km', 'mean')
    )
)

district_summary['Per_10k_People'] = round(district_summary['Facilities'] / district_summary['Population'] * 10000, 2)
district_summary['Avg_Distance'] = round(district_summary['Avg_Distance'], 1)

print("=== TOP 10 BEST SERVED DISTRICTS ===\n")
print(district_summary.sort_values('Per_10k_People', ascending=False).head(10).to_string())

print("\n\n=== TOP 10 WORST SERVED DISTRICTS ===\n")
print(district_summary.sort_values('Per_10k_People', ascending=True).head(10).to_string())

=== TOP 10 BEST SERVED DISTRICTS ===

                                       Facilities Population  Avg_Distance Per_10k_People
District                   Region                                                        
NORTH EAST GONJA           SAVANNAH            64      39404          36.2      16.242006
NANDOM                     UPPER WEST          52      51328           7.4      10.130923
LAWRA                      UPPER WEST          57      58433           8.2       9.754762
BUILSA NORTH               UPPER EAST          51      56571           9.9        9.01522
DAFFIAMA-BUSSIE-ISSA       UPPER WEST          34      38754          17.0       8.773288
BIRIM SOUTH                EASTERN             30      35654           8.5       8.414203
SEKYERE AFRAM PLAINS NORTH ASHANTI             26      32640          30.1       7.965686
BUILSA SOUTH               UPPER EAST          29      36575          10.6       7.928913
SISSALA EAST               UPPER WEST          60      80619  

In [24]:
# How many districts have multiple regions?
mixed = master.groupby('District')['Region'].nunique()
mixed = mixed[mixed > 1]

print(f"Districts with multiple regions: {len(mixed)}\n")

for d in sorted(mixed.index)[:15]:
    regions = master[master['District'] == d]['Region'].unique()
    count_per = master[master['District'] == d].groupby('Region').size()
    print(f"  {d}: {dict(count_per)}")

Districts with multiple regions: 20

  ADA EAST: {'GREATER ACCRA': np.int64(28), 'VOLTA': np.int64(3)}
  ASSIN NORTH: {'CENTRAL': np.int64(40), 'EASTERN': np.int64(3)}
  ASUOGYAMAN: {'EASTERN': np.int64(59), 'VOLTA': np.int64(1)}
  AWUTU SENYA EAST: {'CENTRAL': np.int64(56), 'GREATER ACCRA': np.int64(1)}
  AWUTU SENYA WEST: {'CENTRAL': np.int64(26), 'GREATER ACCRA': np.int64(1)}
  GA EAST: {'EASTERN': np.int64(2), 'GREATER ACCRA': np.int64(85)}
  GA SOUTH: {'CENTRAL': np.int64(1), 'GREATER ACCRA': np.int64(12)}
  GUAN: {'OTI': np.int64(2), 'VOLTA': np.int64(17)}
  KPONE-KATAMANSO: {'EASTERN': np.int64(1), 'GREATER ACCRA': np.int64(77)}
  LOWER MANYA-KROBO: {'EASTERN': np.int64(41), 'GREATER ACCRA': np.int64(1)}
  NSAWAM ADOAGYIRI: {'EASTERN': np.int64(47), 'GREATER ACCRA': np.int64(1)}
  SEKONDI TAKORADI: {'VOLTA': np.int64(1), 'WESTERN': np.int64(55)}
  SEKYERE AFRAM PLAINS NORTH: {'ASHANTI': np.int64(26), 'EASTERN': np.int64(1)}
  SENE WEST: {'AHAFO': np.int64(1), 'BONO EAST': np.int

In [25]:
# Fix: assign each district the region that most of its facilities have
correct_region = master.groupby('District')['Region'].agg(lambda x: x.value_counts().index[0])

# Apply
master['Region'] = master['District'].map(correct_region)

# Also fix GUAN — we know it's OTI
master.loc[master['District'] == 'GUAN', 'Region'] = 'OTI'

# Verify
mixed_after = master.groupby('District')['Region'].nunique()
mixed_after = mixed_after[mixed_after > 1]

print(f"Districts with multiple regions: {len(mixed_after)}")
print(f"\nSpot check:")
for d in ['KPONE-KATAMANSO', 'SEKONDI TAKORADI', 'GA SOUTH', 'WEIJA GBAWE', 'GUAN']:
    r = master[master['District'] == d]['Region'].unique()
    print(f"  {d}: {r}")

Districts with multiple regions: 0

Spot check:
  KPONE-KATAMANSO: ['GREATER ACCRA']
  SEKONDI TAKORADI: ['WESTERN']
  GA SOUTH: ['GREATER ACCRA']
  WEIJA GBAWE: ['GREATER ACCRA']
  GUAN: ['OTI']


In [26]:
# District-level with correct regions AND population
district_summary = (
    master.groupby(['District', 'Region'])
    .agg(
        Facilities=('Name', 'count'),
        Population=('District_Population', 'first'),
        Avg_Distance=('distance_from_centroid_km', 'mean')
    )
)

district_summary['Per_10k_People'] = round(district_summary['Facilities'] / district_summary['Population'] * 10000, 2)
district_summary['Avg_Distance'] = round(district_summary['Avg_Distance'], 1)

print("=== TOP 10 BEST SERVED DISTRICTS ===\n")
print(district_summary.sort_values('Per_10k_People', ascending=False).head(10).to_string())

print("\n\n=== TOP 10 WORST SERVED DISTRICTS ===\n")
print(district_summary.sort_values('Per_10k_People', ascending=True).head(10).to_string())

=== TOP 10 BEST SERVED DISTRICTS ===

                                       Facilities Population  Avg_Distance Per_10k_People
District                   Region                                                        
NORTH EAST GONJA           SAVANNAH            64      39404          36.2      16.242006
NANDOM                     UPPER WEST          52      51328           7.4      10.130923
LAWRA                      UPPER WEST          57      58433           8.2       9.754762
BUILSA NORTH               UPPER EAST          51      56571           9.9        9.01522
DAFFIAMA-BUSSIE-ISSA       UPPER WEST          34      38754          17.0       8.773288
BIRIM SOUTH                EASTERN             30      35654           8.5       8.414203
SEKYERE AFRAM PLAINS NORTH ASHANTI             27      32640          30.2       8.272059
BUILSA SOUTH               UPPER EAST          29      36575          10.6       7.928913
SISSALA EAST               UPPER WEST          60      80619  

In [27]:
master.to_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv", index=False)

print("✅ Master v3 updated with corrected regions")

✅ Master v3 updated with corrected regions


In [28]:
# What facility types does each district have?
# Group into tiers:
# Tier 1 (Basic): CHPS
# Tier 2 (Mid): Health Centre, Clinic, Maternity Home
# Tier 3 (Advanced): Hospital, District Hospital, Polyclinic, Regional Hospital, Teaching Hospital, University Hospital

def facility_tier(ftype):
    if ftype == 'CHPS':
        return 'Basic'
    elif ftype in ['HEALTH CENTRE', 'CLINIC', 'MATERNITY HOME']:
        return 'Mid'
    else:
        return 'Advanced'

master['Facility_Tier'] = master['Facility_Type'].apply(facility_tier)

# Districts with NO advanced facilities
district_tiers = master.groupby('District')['Facility_Tier'].apply(lambda x: set(x))

no_advanced = [d for d, tiers in district_tiers.items() if 'Advanced' not in tiers]
only_basic = [d for d, tiers in district_tiers.items() if tiers == {'Basic'}]

print(f"Total districts: {len(district_tiers)}")
print(f"Districts with NO hospital/polyclinic: {len(no_advanced)}")
print(f"Districts with ONLY CHPS: {len(only_basic)}")

print(f"\n=== Districts with NO advanced facility ===\n")
for d in sorted(no_advanced):
    region = master[master['District'] == d]['Region'].iloc[0]
    count = len(master[master['District'] == d])
    print(f"  {d:<35} {region:<18} {count} facilities (all basic/mid)")

Total districts: 261
Districts with NO hospital/polyclinic: 38
Districts with ONLY CHPS: 0

=== Districts with NO advanced facility ===

  ACHIASE                             EASTERN            20 facilities (all basic/mid)
  ADAKLU                              VOLTA              20 facilities (all basic/mid)
  ADANSI AKROFUOM                     ASHANTI            15 facilities (all basic/mid)
  ADANSI ASOKWA                       ASHANTI            28 facilities (all basic/mid)
  AFIGYA-KWABRE NORTH                 ASHANTI            12 facilities (all basic/mid)
  AGOTIME ZIOPE                       VOLTA              15 facilities (all basic/mid)
  AHAFO-ANO SOUTH WEST                ASHANTI            13 facilities (all basic/mid)
  AKATSI NORTH                        VOLTA              13 facilities (all basic/mid)
  AKWAPEM SOUTH                       EASTERN            25 facilities (all basic/mid)
  ANLOGA                              VOLTA              16 facilities (all basi

38 districts — that's nearly 15% of all districts in Ghana — have zero hospitals, zero polyclinics, zero district hospitals. People in these areas have no access to advanced care without traveling to another district.
Look at some of these:

GA SOUTH in Greater Accra — the capital region! 350,000 people, no hospital
EAST GONJA — only 6 facilities total, all basic/mid, for 117,000 people
WA WEST — 67 facilities but not a single hospital among them
KRACHI EAST — 41 facilities, no hospital

That's a devastating finding. Now let's see which regions are hit hardest:

In [29]:
# Which regions have the most districts without hospitals?
no_adv_regions = master[master['District'].isin(no_advanced)].groupby('Region')['District'].nunique()
total_per_region = master.groupby('Region')['District'].nunique()

comparison = pd.DataFrame({
    'Total_Districts': total_per_region,
    'No_Hospital': no_adv_regions
}).fillna(0).astype(int)

comparison['Percentage'] = round(comparison['No_Hospital'] / comparison['Total_Districts'] * 100, 1)
comparison = comparison.sort_values('Percentage', ascending=False)

print("=== REGIONS: Districts without any hospital ===\n")
print(comparison[comparison['No_Hospital'] > 0].to_string())

=== REGIONS: Districts without any hospital ===

               Total_Districts  No_Hospital  Percentage
Region                                                 
SAVANNAH                     7            2        28.6
EASTERN                     33            8        24.2
OTI                          9            2        22.2
WESTERN NORTH                9            2        22.2
VOLTA                       18            4        22.2
BONO EAST                   11            2        18.2
UPPER WEST                  11            2        18.2
NORTH EAST                   6            1        16.7
BONO                        12            2        16.7
ASHANTI                     43            7        16.3
UPPER EAST                  15            2        13.3
NORTHERN                    16            2        12.5
CENTRAL                     22            1         4.5
GREATER ACCRA               29            1         3.4


In [30]:
# Save the Facility_Tier column too
master.to_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv", index=False)

print("✅ Master v3 saved with Facility_Tier column")

✅ Master v3 saved with Facility_Tier column


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

master = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")

print(f"✅ Loaded: {master.shape[0]} facilities x {master.shape[1]} columns")
print(f"Districts: {master['District'].nunique()}")
print(f"Regions: {master['Region'].nunique()}")

✅ Loaded: 9978 facilities x 30 columns
Districts: 261
Regions: 16


In [2]:
# Who owns the facilities?
print("=== OWNERSHIP BREAKDOWN ===\n")
print(master['Ownership'].value_counts().to_string())

=== OWNERSHIP BREAKDOWN ===

Ownership
GOVERNMENT           7972
PRIVATE              1517
CHAG                  365
QUASI-GOVERNMENT       88
OTHER FAITH-BASED      26
MINES                  10


In [4]:
# Ownership by region
ownership_region = master.groupby(['Region', 'Ownership']).size().unstack(fill_value=0)

# Calculate percentage of non-government facilities per region
ownership_region['Total'] = ownership_region.sum(axis=1)
ownership_region['Govt_Pct'] = round(ownership_region['GOVERNMENT'] / ownership_region['Total'] * 100, 1)
ownership_region['Private_Pct'] = round(ownership_region['PRIVATE'] / ownership_region['Total'] * 100, 1)

print("=== OWNERSHIP BY REGION ===\n")
print(ownership_region[['Total', 'GOVERNMENT', 'PRIVATE', 'CHAG', 'Govt_Pct', 'Private_Pct']].sort_values('Private_Pct', ascending=False).to_string())

=== OWNERSHIP BY REGION ===

Ownership      Total  GOVERNMENT  PRIVATE  CHAG  Govt_Pct  Private_Pct
Region                                                                
GREATER ACCRA   1468         889      523    18      60.6         35.6
ASHANTI         1645        1207      323   101      73.4         19.6
WESTERN          659         506      109    21      76.8         16.5
BONO             478         366       76    28      76.6         15.9
CENTRAL          773         634      111    19      82.0         14.4
BONO EAST        408         345       50    12      84.6         12.3
WESTERN NORTH    324         260       39    23      80.2         12.0
NORTHERN         562         482       52    16      85.8          9.3
VOLTA            527         455       46    22      86.3          8.7
AHAFO            197         167       17    11      84.8          8.6
UPPER EAST       673         597       50    24      88.7          7.4
EASTERN         1125        1004       82    32 

This tells a clear story:
GREATER ACCRA — 35.6% private. Where there's money and people, private healthcare fills the gap.
UPPER WEST, SAVANNAH, OTI — less than 4% private. Nobody is investing in healthcare facilities in these regions. If the government doesn't provide it, nobody does.
So the regions that are already worst served by distance (Savannah at 32km) are also the ones where private sector isn't stepping in.
Now let's check urban vs rural:

In [6]:
# Urban vs Rural comparison
# Split districts into urban-heavy and rural-heavy based on percentage
master['Area_Type'] = master['Percentage of Urban'].apply(
    lambda x: 'Mostly Urban' if x >= 0.5 else 'Mostly Rural'
)

urban_rural = master.groupby('Area_Type').agg(
    Facilities=('Name', 'count'),
    Districts=('District', 'nunique'),
    Avg_Distance=('distance_from_centroid_km', 'mean'),
    Population=('District_Population', lambda x: x.drop_duplicates().sum())
)

urban_rural['Per_10k_People'] = round(urban_rural['Facilities'] / urban_rural['Population'] * 10000, 2)
urban_rural['Avg_Distance'] = round(urban_rural['Avg_Distance'], 1)

print("=== URBAN VS RURAL ===\n")
print(urban_rural.to_string())

=== URBAN VS RURAL ===

              Facilities  Districts  Avg_Distance  Population  Per_10k_People
Area_Type                                                                    
Mostly Rural        5221        155          14.4    14008268            3.73
Mostly Urban        4757        106           7.6    16823751            2.83


In [7]:
# Facility tier by area type
tier_by_area = master.groupby(['Area_Type', 'Facility_Tier']).size().unstack(fill_value=0)
tier_by_area['Total'] = tier_by_area.sum(axis=1)

for tier in ['Basic', 'Mid', 'Advanced']:
    tier_by_area[f'{tier}_Pct'] = round(tier_by_area[tier] / tier_by_area['Total'] * 100, 1)

print("=== FACILITY TIERS: URBAN VS RURAL ===\n")
print(tier_by_area.to_string())

=== FACILITY TIERS: URBAN VS RURAL ===

Facility_Tier  Advanced  Basic   Mid  Total  Basic_Pct  Mid_Pct  Advanced_Pct
Area_Type                                                                    
Mostly Rural        251   3880  1090   5221       74.3     20.9           4.8
Mostly Urban        603   2852  1302   4757       60.0     27.4          12.7


In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)

master = pd.read_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v3.csv")

print(f"✅ Loaded: {master.shape[0]} facilities x {master.shape[1]} columns")

✅ Loaded: 9978 facilities x 30 columns
